In [42]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from astropy.io import fits
from torchvision import transforms

parent_dir = os.path.abspath(os.path.join(os.path.dirname('utils.py'), '..'))
sys.path.append(parent_dir)
import utils as ut  # noqa: E402

In [43]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_FEATURES = ['FLAG']
BANDS = ['f444w-f150w', 'f444w-f277w'] + ['f150w', 'f277w', 'f444w']
SEED = 42
BEST_ACC = 0

# HYPERPARAMETERS
INPUT_SIZE = 67
EPOCHS = 200
NUM_TRIALS = 20
PATIENCE = 20
OPTIMIZE = False

if OPTIMIZE:
    COMMENT = input('Enter OPTIMIZATION comment: ')

In [44]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

if SEED:
    ut.set_seeds(SEED)

## Prepare Data

In [45]:
cwd = Path.cwd()
cutout_dir = cwd.parent / 'data' / 'cutouts'
model_dir = cwd.parent / 'models'

In [46]:
# LOAD DATA INTO DF

cols = {'idx': [], 'images': [], 'header': []}

for folder in cutout_dir.iterdir():
    if folder.is_dir():
        cols['idx'].append(folder.stem)

        temp_df = ut.fits_df(folder)
        temp_df.sort_values(by='BAND')

        # load images
        temp_imgs = {}
        for _, row in temp_df.iterrows():
            hdul = fits.open(row['FILEPATH'])
            data = hdul[0].data.astype('float32')
            header = hdul[0].header
            temp_imgs[row['BAND']] = data
            hdul.close()
        cols['images'].append(temp_imgs)
        cols['header'].append(header)

# turn dict into df and expand header columns
df = pd.DataFrame(cols)
to_expand = ['header', 'images']
for col in to_expand:
    temp_df = pd.DataFrame(df[col].tolist(), index=df.index)
    df = df.join(temp_df)

df.drop(columns=['header', 'BAND', 'images'], inplace=True)

# shuffle
df = df.sample(frac=1, random_state=SEED) if SEED else df.sample(frac=1)

In [47]:
# GET SUBTRACTED AND DIVIDED BANDS

image_cols = ['f150w', 'f277w', 'f444w']

for col in image_cols:
    df[col] = df[col].apply(lambda arr: np.log1p(arr))

numers = ['f150w', 'f277w']
denominator = 'f444w'

for numerator in numers:
    df[numerator + '/' + denominator] = df[numerator] / df[denominator]
    df[denominator + '-' + numerator] = df[denominator] - df[numerator]

In [48]:
# STACK AND RESIZE BANDS

resize_transform = transforms.Resize(
    INPUT_SIZE, transforms.InterpolationMode.BILINEAR
)

df['images'] = df[BANDS].values.tolist()
df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]

In [49]:
# SPLIT INTO TRAIN VAL TEST DFS

train_val_df = df[df['TABLE'] != 'test']
tables = sorted(train_val_df['TABLE'].unique())

train_df = train_val_df[train_val_df['TABLE'].isin(tables[0:-2])]
val_df = train_val_df[~train_val_df['TABLE'].isin(tables[0:-2])]
test_df = df[df['TABLE'] == 'test']

len(train_df), len(val_df), len(test_df)

(342, 96, 48)

In [50]:
# EXTRACT X and y from dfs

X_train = torch.stack(train_df['images'].tolist())
X_val = torch.stack(val_df['images'].tolist())
X_test = torch.stack(test_df['images'].tolist())

y_train = torch.tensor(train_df['FLAG'].tolist())
y_val = torch.tensor(val_df['FLAG'].tolist())
y_test = torch.tensor(test_df['FLAG'].tolist())

i_train = torch.tensor(train_df['idx'].astype(int).tolist())
i_val = torch.tensor(val_df['idx'].astype(int).tolist())
i_test = torch.tensor(test_df['idx'].astype(int).tolist())

In [51]:
X_train_t = X_train.clone().detach()
X_val_t = X_val.clone().detach()
X_test_t = X_test.clone().detach()

In [52]:
# NO SCALING

datasets = {'train': X_train_t, 'val': X_val_t, 'test': X_test_t}

# generate and print tables for each split
for name, data in datasets.items():
    data_np = data.numpy()
    axes = (0, 2, 3)

    means = np.mean(data_np, axis=axes)
    medians = np.median(data_np, axis=axes)
    stds = np.std(data_np, axis=axes)

    # medians = data.median(axis=axes)
    q75, q25 = np.percentile(data_np, [75, 25], axis=axes)
    iqrs = q75 - q25

    maxes = data_np.max(axis=axes)

    df = pd.DataFrame(
        {
            f'{name} mean': means,
            f'{name} std': stds,
            f'{name} median': medians,
            f'{name} iqr': iqrs,
            f'{name} max': maxes,
        }
    )

    print('\n', df.round(4))


    train mean  train std  train median  train iqr  train max
0      0.0729     0.1797        0.0221     0.0720     3.7744
1      0.0331     0.0937        0.0101     0.0416     2.0524
2      0.0369     0.1104        0.0128     0.0460     4.1594
3      0.0766     0.1783        0.0214     0.0589     4.1136
4      0.1098     0.2460        0.0288     0.0819     4.4016

    val mean  val std  val median  val iqr  val max
0    0.0709   0.1791      0.0207   0.0723   2.1785
1    0.0323   0.0918      0.0087   0.0431   1.9178
2    0.0293   0.0889      0.0119   0.0447   3.6024
3    0.0680   0.1643      0.0205   0.0522   3.3560
4    0.1002   0.2368      0.0270   0.0743   3.7504

    test mean  test std  test median  test iqr  test max
0     0.0831    0.2009       0.0241    0.0783    2.3759
1     0.0412    0.1062       0.0118    0.0455    1.2343
2     0.0322    0.0961       0.0120    0.0441    2.4593
3     0.0740    0.1709       0.0211    0.0578    2.7317
4     0.1152    0.2578       0.0301    0.0

In [53]:
X_train_t = X_train.clone().detach()
X_val_t = X_val.clone().detach()
X_test_t = X_test.clone().detach()

In [54]:
# SCALE WITH SKLEARN Robust Scaler

from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

N_train, c, h, w = X_train_t.shape
X_train_t = X_train_t.reshape(-1, c).numpy()
X_train_t = scaler.fit_transform(X_train_t)
X_train_t = torch.from_numpy(X_train_t)
X_train_t = X_train_t.reshape(N_train, c, h, w)

datasets = {'train': X_train_t, 'val': X_val_t, 'test': X_test_t}

# generate and print tables for each split
for name, data in datasets.items():
    if name != 'train':
        N_gals = data.shape[0]
        data = data.reshape(-1, c).numpy()
        data = scaler.transform(data)
        data = torch.from_numpy(data)
        data = data.reshape(N_gals, c, h, w)

    data_np = data.numpy()
    axes = (0, 2, 3)

    means = np.mean(data_np, axis=axes)
    medians = np.median(data_np, axis=axes)
    stds = np.std(data_np, axis=axes)

    # medians = data.median(axis=axes)
    q75, q25 = np.percentile(data_np, [75, 25], axis=axes)
    iqrs = q75 - q25

    maxes = data_np.max(axis=axes)

    df = pd.DataFrame(
        {
            f'{name} mean': means,
            f'{name} std': stds,
            f'{name} median': medians,
            f'{name} iqr': iqrs,
            f'{name} max': maxes,
        }
    )

    print('\n', df.round(4))


    train mean  train std  train median  train iqr  train max
0      0.9544     3.1483        0.0644     1.2615  65.853996
1      0.2576     1.6419       -0.1457     0.7289  35.650002
2      0.3236     1.9338       -0.0987     0.8058  72.604202
3      1.0204     3.1248        0.0522     1.0312  71.801003
4      1.6005     4.3098        0.1822     1.4350  76.822304

    val mean  val std  val median  val iqr    val max
0    0.9199   3.1381      0.0400   1.2669  37.853100
1    0.2426   1.6079     -0.1693   0.7543  33.290600
2    0.1915   1.5582     -0.1141   0.7836  62.814602
3    0.8688   2.8785      0.0361   0.9146  58.438599
4    1.4339   4.1484      0.1512   1.3013  65.399902

    test mean  test std  test median  test iqr   test max
0     1.1328    3.5195       0.0994    1.3719  41.291500
1     0.3998    1.8611      -0.1154    0.7966  21.296200
2     0.2408    1.6838      -0.1120    0.7722  42.739300
3     0.9738    2.9952       0.0470    1.0126  47.507801
4     1.6962    4.5169   

In [55]:
X_train_t = X_train.clone().detach()
X_val_t = X_val.clone().detach()
X_test_t = X_test.clone().detach()

In [56]:
# PYTORCH NORMALIZE

from torchvision.transforms import Normalize

tmeans = X_train_t.mean(axis=axes)
tstds = X_train_t.std(axis=axes)

scaler = Normalize(mean=tmeans, std=tstds)

datasets = {'train': X_train_t, 'val': X_val_t, 'test': X_test_t}

# generate and print tables for each split
for name, data in datasets.items():
    data = scaler(data)
    data_np = data.numpy()
    axes = (0, 2, 3)

    means = np.mean(data_np, axis=axes)
    medians = np.median(data_np, axis=axes)
    stds = np.std(data_np, axis=axes)

    # medians = data.median(axis=axes)
    q75, q25 = np.percentile(data_np, [75, 25], axis=axes)
    iqrs = q75 - q25

    maxes = data_np.max(axis=axes)

    df = pd.DataFrame(
        {
            f'{name} mean': means,
            f'{name} std': stds,
            f'{name} median': medians,
            f'{name} iqr': iqrs,
            f'{name} max': maxes,
        }
    )

    print('\n', df.round(4))


    train mean  train std  train median  train iqr  train max
0        -0.0        1.0       -0.2827     0.4007  20.600401
1        -0.0        1.0       -0.2456     0.4439  21.549299
2        -0.0        1.0       -0.2184     0.4167  37.353500
3         0.0        1.0       -0.3099     0.3300  22.636600
4         0.0        1.0       -0.3291     0.3330  17.448500

    val mean  val std  val median  val iqr    val max
0   -0.0110   0.9968     -0.2904   0.4025  11.718300
1   -0.0091   0.9793     -0.2600   0.4595  20.112700
2   -0.0683   0.8058     -0.2263   0.4052  32.306099
3   -0.0485   0.9212     -0.3150   0.2927  18.388300
4   -0.0387   0.9626     -0.3363   0.3019  14.801100

    test mean  test std  test median  test iqr  test max
0     0.0567    1.1179      -0.2716    0.4357   12.8172
1     0.0866    1.1335      -0.2272    0.4852   12.8185
2    -0.0428    0.8707      -0.2253    0.3993   21.9496
3    -0.0149    0.9585      -0.3115    0.3241   14.8878
4     0.0222    1.0480      -0